# 09 — S1: Decoding strategies para reducir mode collapse

Este notebook analiza los outputs del script:

```bash
scripts/run_s1_decoding_experiment.py
```

Objetivo de S1:

- Probar alternativas de decoding sin reentrenar.
- Comparar `greedy`, sampling con temperature/nucleus, diverse beam y contrastive decoding.
- Medir si mejora la diversidad textual.
- Evaluar si esa diversidad mantiene coherencia clínica.

Este notebook no genera captions directamente. Solo lee outputs, grafica y ayuda a interpretar resultados.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "decoding_sampling" / "s1_selected_30"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 1. Comandos S1

Cuando tengas el checkpoint fine-tuneado real, el comando principal será:

In [ ]:
cmd_real = f"""
cd "{PROJECT_ROOT}"

python scripts/run_s1_decoding_experiment.py \\
  --model-dir models/blip_finetuned_5k/best \\
  --indices data/selected_indices.json \\
  --max-images 30 \\
  --device cpu
"""

print(cmd_real)

Para verificar paths sin correr inferencia:

In [ ]:
cmd_dry_run = f"""
cd "{PROJECT_ROOT}"

python scripts/run_s1_decoding_experiment.py \\
  --model-dir models/blip_finetuned_5k/best \\
  --indices data/selected_indices.json \\
  --max-images 30 \\
  --device cpu \\
  --dry-run
"""

print(cmd_dry_run)

Smoke test opcional con checkpoint debug actual. No sirve como resultado final, pero permite verificar que el script corre de punta a punta:

In [ ]:
cmd_smoke = f"""
cd "{PROJECT_ROOT}"

python scripts/run_s1_decoding_experiment.py \\
  --allow-debug \\
  --indices data/selected_indices.json \\
  --max-images 2 \\
  --samples-per-image 2 \\
  --device cpu
"""

print(cmd_smoke)

## 2. Verificación de outputs

El script S1 debería generar:

```text
outputs/decoding_sampling/s1_selected_30/s1_all_captions.csv
outputs/decoding_sampling/s1_selected_30/s1_decoding_summary.csv
outputs/decoding_sampling/s1_selected_30/s1_image_strategy_summary.csv
outputs/decoding_sampling/s1_selected_30/s1_examples.csv
outputs/decoding_sampling/s1_selected_30/s1_errors.csv
outputs/decoding_sampling/s1_selected_30/s1_strategy_comparison.png
```

In [ ]:
paths = {
    "all_captions": OUTPUT_DIR / "s1_all_captions.csv",
    "summary": OUTPUT_DIR / "s1_decoding_summary.csv",
    "image_summary": OUTPUT_DIR / "s1_image_strategy_summary.csv",
    "examples": OUTPUT_DIR / "s1_examples.csv",
    "errors": OUTPUT_DIR / "s1_errors.csv",
    "plot": OUTPUT_DIR / "s1_strategy_comparison.png",
}

for name, path in paths.items():
    print(f"{name:<15}", "OK" if path.exists() else "FALTA", path)

In [ ]:
missing = [name for name, path in paths.items() if name != "plot" and not path.exists()]

if missing:
    print("Todavía faltan outputs de S1:", missing)
    print("Esto es normal si todavía no corriste scripts/run_s1_decoding_experiment.py.")
else:
    captions_df = pd.read_csv(paths["all_captions"])
    summary_df = pd.read_csv(paths["summary"])
    image_summary_df = pd.read_csv(paths["image_summary"])
    examples_df = pd.read_csv(paths["examples"])
    errors_df = pd.read_csv(paths["errors"])

    print("captions_df:", captions_df.shape)
    print("summary_df:", summary_df.shape)
    print("image_summary_df:", image_summary_df.shape)
    print("examples_df:", examples_df.shape)
    print("errors_df:", errors_df.shape)

## 3. Resumen por estrategia

Métricas clave:

- `n_unique`: cantidad de captions únicas.
- `unique_ratio`: proporción de captions únicas.
- `top_pct`: porcentaje ocupado por la caption más repetida.
- `pct_normal`: proporción de captions que parecen normales/genéricas.
- `pct_specific_clinical`: proporción de captions con alguna keyword clínica específica.

In [ ]:
if not missing:
    cols = [
        "strategy",
        "n_images",
        "n_captions",
        "n_unique",
        "unique_ratio",
        "top_pct",
        "mean_len_words",
        "pct_normal",
        "pct_specific_clinical",
        "top_caption_norm",
    ]
    existing_cols = [c for c in cols if c in summary_df.columns]
    display(summary_df[existing_cols])

## 4. Gráfico principal: diversidad vs repetición

Una estrategia buena debería subir `unique_ratio` y bajar `top_pct`.

In [ ]:
if not missing:
    ax = summary_df.plot(
        x="strategy",
        y=["unique_ratio", "top_pct"],
        kind="bar",
        figsize=(10, 4),
    )

    ax.set_title("S1 — Diversidad vs repetición por estrategia")
    ax.set_ylabel("Proporción")
    ax.set_xlabel("Estrategia")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()

    fig_path = OUTPUT_DIR / "s1_notebook_diversity_vs_repetition.png"
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Figura guardada en:", fig_path)

## 5. Gráfico clínico: normalidad vs especificidad

Este gráfico ayuda a ver si la diversidad nueva es útil o si solo agrega ruido.

In [ ]:
if not missing:
    clinical_cols = [c for c in ["pct_normal", "pct_specific_clinical", "pct_other"] if c in summary_df.columns]

    if clinical_cols:
        ax = summary_df.plot(
            x="strategy",
            y=clinical_cols,
            kind="bar",
            figsize=(10, 4),
        )

        ax.set_title("S1 — Tipo de contenido generado por estrategia")
        ax.set_ylabel("Proporción")
        ax.set_xlabel("Estrategia")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()

        fig_path = OUTPUT_DIR / "s1_notebook_clinical_content.png"
        plt.savefig(fig_path, dpi=200)
        plt.show()

        print("Figura guardada en:", fig_path)
    else:
        print("No hay columnas clínicas para graficar.")

## 6. Captions por estrategia

Vista general de las captions generadas. Útil para inspección cualitativa.

In [ ]:
if not missing:
    cols = [
        "model_tag",
        "strategy",
        "idx",
        "sample_id",
        "reference",
        "caption",
        "categories",
        "len_words",
    ]
    existing_cols = [c for c in cols if c in captions_df.columns]
    display(captions_df[existing_cols].head(80))

## 7. Ejemplos cualitativos

El script guarda ejemplos iniciales y ejemplos de la caption dominante por estrategia.

In [ ]:
if not missing:
    display(examples_df.head(80))

## 8. Comparación por imagen

Permite revisar una imagen puntual y ver qué produjo cada estrategia.

In [ ]:
if not missing:
    available_indices = sorted(captions_df["idx"].unique().tolist())
    print("Índices disponibles:", available_indices)

    idx_to_show = available_indices[0]
    print("idx_to_show:", idx_to_show)

In [ ]:
if not missing:
    idx_view = captions_df[captions_df["idx"] == idx_to_show].copy()

    cols = [
        "strategy",
        "sample_id",
        "reference",
        "caption",
        "categories",
        "len_words",
    ]
    existing_cols = [c for c in cols if c in idx_view.columns]
    display(idx_view[existing_cols])

## 9. Casos donde una estrategia mejora diversidad

Ordenamos por imagen y estrategia para buscar dónde una estrategia generó más variedad.

In [ ]:
if not missing:
    cols = [
        "strategy",
        "idx",
        "n_captions",
        "n_unique",
        "unique_ratio",
        "top_pct",
        "reference",
        "captions_joined",
    ]
    existing_cols = [c for c in cols if c in image_summary_df.columns]

    diverse_cases = image_summary_df.sort_values(
        ["unique_ratio", "n_unique"],
        ascending=[False, False],
    )

    display(diverse_cases[existing_cols].head(30))

## 10. Casos donde sigue habiendo collapse

Estos casos mantienen `top_pct` alto incluso con sampling/diverse decoding.

In [ ]:
if not missing:
    cols = [
        "strategy",
        "idx",
        "n_captions",
        "n_unique",
        "unique_ratio",
        "top_pct",
        "reference",
        "captions_joined",
    ]
    existing_cols = [c for c in cols if c in image_summary_df.columns]

    collapsed_cases = image_summary_df.sort_values(
        ["top_pct", "unique_ratio"],
        ascending=[False, True],
    )

    display(collapsed_cases[existing_cols].head(30))

## 11. Errores de generación

Algunas estrategias pueden fallar dependiendo de la versión de `transformers`.

In [ ]:
if not missing:
    if errors_df.empty:
        print("No hubo errores registrados.")
    else:
        display(errors_df)

## 12. Interpretación automática preliminar

Esta celda propone una lectura inicial de cada estrategia.

In [ ]:
if not missing:
    for _, row in summary_df.iterrows():
        strategy = row["strategy"]
        unique_ratio = row["unique_ratio"]
        top_pct = row["top_pct"]
        pct_specific = row.get("pct_specific_clinical", None)
        pct_normal = row.get("pct_normal", None)

        print("=" * 100)
        print("Estrategia:", strategy)
        print(f"unique_ratio: {unique_ratio:.3f}")
        print(f"top_pct: {top_pct:.3f}")

        if pct_specific is not None:
            print(f"pct_specific_clinical: {pct_specific:.3f}")
        if pct_normal is not None:
            print(f"pct_normal: {pct_normal:.3f}")

        print()

        if unique_ratio >= 0.50 and top_pct <= 0.30:
            print("Lectura preliminar: buena mejora de diversidad.")
            print("Revisar manualmente si las captions siguen siendo clínicamente razonables.")
        elif unique_ratio > 0.25 and top_pct < 0.60:
            print("Lectura preliminar: mejora parcial.")
            print("Puede servir como alternativa si no introduce ruido clínico excesivo.")
        else:
            print("Lectura preliminar: sigue habiendo collapse fuerte.")
            print("Probablemente el problema no se resuelva solo con decoding.")

        print()

## 13. Conclusión para informe

Completar después de correr S1 real:

- ¿Qué estrategia maximiza diversidad?
- ¿Cuál reduce más la caption dominante?
- ¿La diversidad nueva es clínicamente útil o introduce ruido?
- ¿S1 alcanza para usar captions finales, o solo se reporta como intento de mitigación?

Conclusión preliminar:

In [ ]:
conclusion = '''
Pendiente de completar después de correr S1 real.

Posibles lecturas:

1. Si sampling aumenta diversidad sin perder coherencia:
   Se puede usar esa estrategia para captions finales y reportar que greedy amplificaba el collapse.

2. Si sampling aumenta diversidad pero con captions incoherentes:
   S1 muestra un trade-off: más variedad, menor confiabilidad clínica.

3. Si ninguna estrategia mejora:
   El collapse está internalizado en el modelo, no es solo un problema de greedy decoding.
   En ese caso, el siguiente paso sería más datos o balanceo semántico.

4. Si diverse beam mejora más que sampling:
   El modelo tiene alternativas plausibles, pero necesita una búsqueda que fuerce diversidad.
'''
print(conclusion)